## Process objects-raw.json to give objects.json

In [ ]:
import json

objectdict = json.load(open("objects-raw.json", "r"))

for id in objectdict.keys():
    valuesdictarray = objectdict[id]
    valuesdict = dict()
    for i in range(len(valuesdictarray)):
        valuesdict |= valuesdictarray[i]
    valuesdict.__delitem__("Provenance")
    objectdict[id] = valuesdict

json.dump(objectdict, open("objects.json", "w"), indent=2)

## Make summary tables

In [ ]:
import json
import sys
import contextlib

objects = json.load(open("objects.json", "r"))

tableheader = r"""
\begin{table}[t]
    \setlength{\tabcolsep}{2pt}
    \footnotesize
    \begin{tabularx}{\linewidth}{lccX}
        \hline
        Name&Position&Type&Other Names\\
        \hline
"""

tablefooter = r"""
        \hline
    \end{tabularx}
\end{table}
\clearpage
"""

nlines = 50

def maketable(thiscatalog, ifirst, ilast, exceptions=[]):
    with open("%s-table.tex" % thiscatalog, "w") as f:
        iline = 0
        with contextlib.redirect_stdout(f):
            for i in range(ifirst, ilast + 1):
                if i in exceptions:
                    continue
                if iline % nlines == 0:
                    print(tableheader)
                for id in objects:
                    object = objects[id]
                    if "Catalog %s" % thiscatalog not in object:
                        continue
                    if object["Catalog %s" % thiscatalog] != i:
                        continue
                    if thiscatalog in ["M", "C", "U"]:
                        thisname = "%s%d" % (thiscatalog, object["Catalog %s" % thiscatalog])
                    else:
                        thisname = "%s %d" % (thiscatalog, object["Catalog %s" % thiscatalog])
                    othercatalogs = ["NGC", "IC", "M", "C", "U", "BSC", "Mel", "Cr"]
                    othernames = []
                    for othercatalog in othercatalogs:
                        if (
                            thiscatalog != othercatalog
                            and "Catalog %s" % othercatalog in object
                        ):
                            if othercatalog in ["M", "C", "U"]:
                                othernames.append(
                                    "%s%s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                            else:
                                othernames.append(
                                    "%s %s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                    if "Name" in object:
                        othernames.append(object["Name"])
                    othernames = " = ".join(othernames)
                    line = "%-4s &$%04.1f$ $%+03.0f$&%-3s&%s\\\\" % (
                        thisname,
                        object["alpha"] / 15,
                        object["delta"],
                        object["Type"],
                        othernames,
                    )
                    print(line, file=sys.stderr)
                    print(line)
                    iline += 1
                    if iline % nlines == 0:
                        print(tablefooter)
            if iline % nlines != 0:
                print(tablefooter)

maketable("M", 1, 110)
maketable("C", 1, 109)
maketable("U", 1, 87)
maketable("BSC", 1, 150, [2, 8, 71])

## Make Bodies

In [ ]:
import json
import sys
import contextlib

objects = json.load(open("objects.json", "r"))

tableheader = r"""
\begin{table}[t]
    \setlength{\tabcolsep}{2pt}
    \footnotesize
    \begin{tabularx}{\linewidth}{lccX}
        \hline
        Name&Position&Type&Other Names\\
        \hline
"""

tablefooter = r"""
        \hline
    \end{tabularx}
\end{table}
\clearpage
"""

def makebody(thiscatalog, ifirst, ilast):
    with open("%s-body.tex" % thiscatalog, "w") as f:
        iline = 0
        with contextlib.redirect_stdout(f):
            for i in range(ifirst, ilast + 1):
                for id in objects:

                    object = objects[id]

                    if "Catalog %s" % thiscatalog not in object:
                        continue
                    if object["Catalog %s" % thiscatalog] != i:
                        continue

                    thisname = "%s%d" % (thiscatalog, object["Catalog %s" % thiscatalog])

                    othercatalogs = ["NGC", "IC", "M", "C", "U", "BSC", "Mel", "Cr"]
                    othernames = []
                    for othercatalog in othercatalogs:
                        if (
                            thiscatalog != othercatalog
                            and "Catalog %s" % othercatalog in object
                        ):
                            if othercatalog in ["M", "C", "U"]:
                                othernames.append(
                                    "%s%s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                            else:
                                othernames.append(
                                    "%s %s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                    if "Name" in object:
                        othernames.append(object["Name"])
                    othernames = " = ".join(othernames)

                    X = object["X"]
                    if "Y" not in object:
                        Y = 0
                    else:
                        Y = object["Y"]
                    if X >= 180:
                        if X == Y or Y == 0:
                            size = r"\degrees{%.1f}" % (X / 60)
                        else:
                            size = r"\degrees{%.1f} \times \degrees{%.1f}" % (X / 60, Y / 60)
                    else:
                        if X == Y or Y == 0:
                            size = r"\arcmin{%.1f}" % (X)
                        else:
                            size = r"\arcmin{%.1f} \times \arcmin{%.1f}" % (X, Y)

                    psa = object["PSA"]
                    if "OSA" in object:
                        osa = object["OSA"]
                    else:
                        osa = ""
                    if "CSA" in object:
                        csa = object["CSA"]
                    else:
                        csa = ""
                    if "Constellation" in object:
                        constellation = object["Constellation"]
                    else:
                        constellation = ""
                    if "Mag" in object:
                        mag = object["Mag"]
                    else:
                        mag = ""

                    line = r"\chart{%s}{\chartdata{%04.1f}{%+03.0f}{%s}{%s}{%s}{%s}{%s}{%s}{%s}}{}{}{}" % (
                        thisname,
                        object["alpha"] / 15,
                        object["delta"],
                        constellation,
                        psa,
                        osa,
                        csa,
                        object["Type"],
                        mag,
                        size
                    )
                    print(line, file=sys.stderr)
                    print(line)

makebody("M", 1, 110)
makebody("C", 1, 109)
makebody("U", 1, 87)